# Prefiguring the Carceral-Digital Turn: A Computational Linguistic Analysis of the American Prison Writing Archive (APWA)

**Author:** Chayne Wild (Alliance for Social, Political, Ethical, and Cultural Thought [ASPECT], Virginia Tech)  
**Repository:** [https://github.com/chayne-wild/APWA-Scrape-and-STM](https://github.com/chayne-wild/APWA-Scrape-and-STM)

---

### Overview & Sequential Pipeline Architecture
This notebook executes the complete end-to-end replication pipeline sequentially (`00` $\rightarrow$ `09`) in **Google Colaboratory** using the `rpy2` (`%%R`) bridge:
1. **Script 00 (`00_colab_environment_setup.py`):** Mounts Google Drive, enforces workspace isolation (`/content/drive/MyDrive/colab_apwa_folder`), activates `rpy2.ipython`, and installs required R packages.
2. **Script 01 (`01_index_apwa_urls.py`):** Paginates through all 181 archive pages to collect all unique essay URLs into `apwa_master_urls.csv`.
3. **Script 02 (`02_scrape_apwa_archive.py`):** Concurrently scrapes raw transcripts and metadata blocks into `apwa_parsed_archive.jsonl` and logs untranscribed scan/PDF pages to `apwa_excluded_urls.csv`.
4. **Script 03 (`03_tidy_preprocessing.R`):** Fault-tolerant JSONL ingestion in R, 4-digit year extraction (`clean_year`), `tidytext` tokenization, `textstem` lemmatization, the "Plural Bypass" custom stop-word filter, and STM document-term matrix construction.
5. **Script 04 (`04_stm_searchK_diagnostics.R`):** Hyperparameter tuning across $K \in \{10, 20, 30, 40, 50, 60\}$ to empirically justify $K = 40$.
6. **Script 05 (`05_stm_model_training_k40.R`):** Trains the deterministic $K = 40$ Structural Topic Model (`init.type = "Spectral"`) with `~ s(clean_year)` prevalence prior.
7. **Script 06 (`06_qualitative_validation.R`):** Extracts `Highest Prob`, `FREX`, `Lift`, and `Score` word profiles (`labelTopics`) and top full-text exemplar essays (`findThoughts`).
8. **Script 07 (`07_temporal_regressions.R`):** Estimates B-spline temporal trajectories (`estimateEffect`) and Surge-Era OLS regressions (2010–2021 vs. 2016–2021).
9. **Script 08 (`08_pandemic_catalyst.R`):** Models the Pandemic "Ratchet Effect" (Topic 32 Utility vs. Topic 21 COVID-19) and document-level $\theta$ Pearson exclusivity.
10. **Script 09 (`09_network_analysis_igraph.R`):** Computes the 40-topic correlation network (`topicCorr` & `igraph`), isolating the Topic 18 $\leftrightarrow$ Topic 26 "Economy of Scarcity" tie.
11. **Artifact Organization & Reproducibility Log (`run_pipeline.py`):** Routes generated files into `data/`, `results/charts/`, and `results/figures/` and exports `results/session_info.txt`.

## 00. Google Colab Workspace & R-Magic Environment Setup
*Source: `scripts/00_colab_environment_setup.py`*

In [ ]:
# ==============================================================================
# SCRIPT 00 (CELL 0A): MOUNT DRIVE & LOCK SAFE WORKSPACE (PYTHON)
# Task: Mount Google Drive and enforce strict directory isolation.
# ==============================================================================
#
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define and enforce strict workspace isolation
safe_workspace = '/content/drive/MyDrive/colab_apwa_folder'
os.makedirs(safe_workspace, exist_ok=True)
os.chdir(safe_workspace)

# 3. Hard security validation check
if os.getcwd() != safe_workspace:
    raise RuntimeError(
        f"SECURITY HALT: Execution environment compromised. "
        f"Current directory is {os.getcwd()}, expected {safe_workspace}."
    )

print(f"Workspace secured. Active directory: {os.getcwd()}")

In [ ]:
# ==============================================================================
# SCRIPT 00 (CELL 0B): ACTIVATE R-MAGIC BRIDGE IN COLAB (PYTHON)
# ==============================================================================

%load_ext rpy2.ipython
print("R-Magic activated. Begin all subsequent R cells with %%R.")

In [ ]:
%%R
# ==============================================================================
# SCRIPT 00 (CELL 0C): INSTALL REQUIRED R PACKAGES (R)
# Note: Only needs to be run once per fresh Google Colab runtime reset.
# ==============================================================================

cat("Installing Tidyverse, NLP, STM, and visualization packages in R...\n")

required_packages <- c(
  "tidyverse",      # Core Tidy data wrangling and ggplot2
  "jsonlite",       # JSONL parsing
  "tidytext",       # Tidy tokenization and stop-word dictionaries
  "textstem",       # Word lemmatization
  "stm",            # Structural Topic Modeling
  "tm",             # Required dependency for stm::textProcessor
  "SnowballC",      # Required dependency for stm::textProcessor
  "igraph",         # Network graph modeling for topicCorr
  "broom",          # Tidy regression outputs
  "modelsummary",   # Publication-ready regression tables
  "gt"              # Grammar of Tables for manuscript export
)

new_packages <- required_packages[!(required_packages %in% installed.packages()[, "Package"])]
if (length(new_packages) > 0) {
  install.packages(new_packages, quiet = TRUE, dependencies = TRUE)
}

cat("All R dependencies successfully installed and verified.\n")

## 01. Index APWA Archive URLs (Python)
*Source: `scripts/01_index_apwa_urls.py`*  
Paginates through all 181 pages of the APWA web archive to collect and save all unique essay URLs into `apwa_master_urls.csv`. Includes state-tracking (resume capability) and polite rate-limiting.

In [ ]:
# ==============================================================================
# SCRIPT 01: INDEX APWA ARCHIVE URLS (PYTHON)
# Task: Paginate through all 181 pages of the APWA web archive to collect and
#       save all 4,504 unique essay URLs into 'apwa_master_urls.csv'.
#       Includes state-tracking (resume capability) and polite rate-limiting.
# ==============================================================================

import os
import time
import random
import requests
import csv
from bs4 import BeautifulSoup

# 1. Strict Workspace Verification
safe_workspace = '/content/drive/MyDrive/colab_apwa_folder'
os.makedirs(safe_workspace, exist_ok=True)
os.chdir(safe_workspace)

if os.getcwd() != safe_workspace:
    raise RuntimeError(f"SECURITY HALT: Current directory is {os.getcwd()}, expected {safe_workspace}.")

# 2. Scraper Configuration & Polite Academic Identification Header
master_csv = 'apwa_master_urls.csv'
base_url = 'https://prisonwitness.org/the-archive/'
max_pages = 181
headers = {
    'User-Agent': 'APWA_Virginia_Tech_Research_Bot/1.0 (chayne@vt.edu)'
}

# 3. State Tracking (Resume Capability if Colab disconnects)
processed_pages = set()

if os.path.exists(master_csv):
    with open(master_csv, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        next(reader, None)  # Skip CSV header
        for row in reader:
            if row and row[0].isdigit():
                processed_pages.add(int(row[0]))
    print(f"Resuming index creation: {len(processed_pages)}/{max_pages} pages already indexed.")
else:
    with open(master_csv, 'w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['source_page', 'essay_url'])
    print("Initialized new master URL index: 'apwa_master_urls.csv'.")

# 4. Paginated Crawling Loop
for page in range(1, max_pages + 1):
    if page in processed_pages:
        continue

    current_url = f"{base_url}{page}/" if page > 1 else base_url

    try:
        response = requests.get(current_url, headers=headers, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, 'html.parser')

        # Isolate all anchor tags linking to individual APWA essays
        page_urls = set()
        for link in soup.find_all('a', href=True):
            href = link['href']
            if '/apwa-essay/' in href:
                page_urls.add(href)

        # Immediately append discovered URLs to disk
        with open(master_csv, 'a', encoding='utf-8', newline='') as f:
            writer = csv.writer(f)
            for essay_url in sorted(page_urls):
                writer.writerow([page, essay_url])

        print(f"Page {page}/{max_pages} indexed: Found {len(page_urls)} unique essay URLs.")

        # Polite randomized throttling (1.5 to 3.0 seconds) to respect JHU servers
        time.sleep(random.uniform(1.5, 3.0))

    except requests.exceptions.RequestException as e:
        print(f"Network error on page {page}: {e}")
        print("Halting index scrape to prevent data gaps. Rerun script to resume.")
        break
    except Exception as e:
        print(f"Unexpected error parsing page {page}: {e}")
        break

print(f"\n[TASK 01 COMPLETE] Master URL index saved to: {os.path.join(safe_workspace, master_csv)}")

## 02. Scrape APWA Essay Transcripts & Metadata to JSONL (Python)
*Source: `scripts/02_scrape_apwa_archive.py`*  
Reads `apwa_master_urls.csv`, concurrently scrapes each essay page using DOM structural heuristics (`\nDetails\n` and `\nTranscript\n`), writes raw JSON records to `apwa_parsed_archive.jsonl`, and logs untranscribed PDF/Image-only pages to `apwa_excluded_urls.csv`.

In [ ]:
# ==============================================================================
# SCRIPT 02: SCRAPE APWA ESSAY TRANSCRIPTS & METADATA TO JSONL (PYTHON)
# Task: Read 'apwa_master_urls.csv', concurrently scrape each essay page using
#       DOM structural heuristics ('\nDetails\n' and '\nTranscript\n'), write
#       raw JSON records to 'apwa_parsed_archive.jsonl', and log untranscribed
#       PDF/Image-only pages to 'apwa_excluded_urls.csv'.
# Note: ZERO text cleaning or temporal regexing is performed here.
# ==============================================================================

import os
import requests
import json
import csv
import threading
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed

# 1. Strict Workspace Verification
safe_workspace = '/content/drive/MyDrive/colab_apwa_folder'
os.makedirs(safe_workspace, exist_ok=True)
os.chdir(safe_workspace)

if os.getcwd() != safe_workspace:
    raise RuntimeError(f"SECURITY HALT: Current directory is {os.getcwd()}, expected {safe_workspace}.")

# 2. File & Concurrency Configuration
master_csv = 'apwa_master_urls.csv'
output_file = 'apwa_parsed_archive.jsonl'
excluded_csv = 'apwa_excluded_urls.csv'

headers = {
    'User-Agent': 'APWA_Virginia_Tech_Research_Bot/1.0 (chayne@vt.edu)'
}

MAX_WORKERS = 15  # Concurrent threads (reduce if server returns HTTP 429)
BATCH_SIZE = 50   # Number of records buffered before flushing to Google Drive

# Thread-safe locks and memory buffers
write_lock = threading.Lock()
json_buffer = []
excluded_buffer = []
success_count = 0
excluded_count = 0

def flush_buffers():
    """Flushes in-memory buffers to disk in a single thread-safe I/O operation."""
    global json_buffer, excluded_buffer

    if json_buffer:
        with open(output_file, 'a', encoding='utf-8') as f_out:
            f_out.write('\n'.join(json_buffer) + '\n')
            f_out.flush()
        json_buffer.clear()

    if excluded_buffer:
        with open(excluded_csv, 'a', encoding='utf-8', newline='') as f_ex:
            writer = csv.writer(f_ex)
            writer.writerows(excluded_buffer)
            f_ex.flush()
        excluded_buffer.clear()

def parse_apwa_page(url):
    """
    Fetches an APWA essay page and isolates typed transcripts from metadata
    using explicit DOM string-partitioning boundaries.
    """
    try:
        response = requests.get(url, headers=headers, timeout=15)
        if response.status_code == 429:
            print(f"RATE LIMITED (429) on {url}")
            return (url, None)

        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')

        # Strip non-text script and style elements
        for element in soup(["script", "style"]):
            element.extract()

        page_text = soup.get_text(separator='\n', strip=True)

        # Heuristic 1: Isolate the Metadata block
        if '\nDetails\n' in page_text:
            main_content, _, metadata_block = page_text.partition('\nDetails\n')
        else:
            return (url, None)

        # Heuristic 2: Isolate the typed Transcript (skips scan/PDF-only pages)
        if '\nTranscript\n' in main_content:
            _, _, transcript_text = main_content.partition('\nTranscript\n')
        else:
            return (url, None)

        transcript_text = transcript_text.strip()
        metadata_block, _, _ = metadata_block.partition('\nIf this is your essay')
        meta_lines = [line.strip() for line in metadata_block.split('\n') if line.strip()]

        # Pair alternating metadata keys and values into a raw dictionary
        metadata = {}
        for i in range(0, len(meta_lines) - 1, 2):
            metadata[meta_lines[i]] = meta_lines[i + 1]

        return (url, {
            'url': url,
            'slug': url.strip('/').split('/')[-1],
            'transcript': transcript_text,
            'metadata': metadata
        })

    except Exception:
        return (url, None)

# 3. State Synchronization & Execution Block
if __name__ == "__main__":
    if not os.path.exists(master_csv):
        raise FileNotFoundError(f"Master CSV '{master_csv}' not found. Run Script 01 first.")

    processed_urls = set()

    # Load previously scraped valid URLs
    if os.path.exists(output_file):
        with open(output_file, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    data = json.loads(line)
                    if 'url' in data:
                        processed_urls.add(data['url'])
                except json.JSONDecodeError:
                    continue

    # Load previously excluded URLs
    if os.path.exists(excluded_csv):
        with open(excluded_csv, 'r', encoding='utf-8') as f:
            reader = csv.reader(f)
            next(reader, None)
            for row in reader:
                if row:
                    processed_urls.add(row[0])
    else:
        with open(excluded_csv, 'w', encoding='utf-8', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['excluded_url', 'reason'])

    # Identify remaining URLs to scrape
    pending_urls = []
    with open(master_csv, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        next(reader, None)
        for row in reader:
            if row and len(row) > 1:
                essay_url = row[1]
                if essay_url not in processed_urls:
                    pending_urls.append(essay_url)

    # Deduplicate pending list while preserving order
    pending_urls = list(dict.fromkeys(pending_urls))
    total_pending = len(pending_urls)

    print(f"State sync complete: {len(processed_urls)} URLs already processed.")
    print(f"Launching concurrent scrape on {total_pending} remaining URLs ({MAX_WORKERS} workers)...")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(parse_apwa_page, url): url for url in pending_urls}

        for i, future in enumerate(as_completed(futures), 1):
            url, parsed_data = future.result()

            with write_lock:
                if parsed_data:
                    json_buffer.append(json.dumps(parsed_data))
                    success_count += 1
                else:
                    excluded_buffer.append([url, 'No Transcript or Parse Error'])
                    excluded_count += 1

                if (len(json_buffer) + len(excluded_buffer)) >= BATCH_SIZE:
                    flush_buffers()
                    print(f"Progress: [{i}/{total_pending}] processed and flushed to Drive.")

    # Final flush for remaining buffered items
    flush_buffers()
    print(f"\n[TASK 02 COMPLETE] Saved: {success_count} | Excluded: {excluded_count}")
    print("Python data acquisition phase finished. Proceed to R scripts (03–09).")

## 03. Tidyverse Data Ingestion, Temporal Normalization, & NLP Cleaning (R)
*Source: `scripts/03_tidy_preprocessing.R`*  
Ingests `apwa_parsed_archive.jsonl` natively in R with fault tolerance (`purrr::safely`), normalizes `clean_year`, executes `tidytext` tokenization, `textstem` lemmatization, and the "Plural Bypass" stop-word filter, and builds the synchronized STM mathematical matrix (`out`).

In [ ]:
%%R
# ==============================================================================
# SCRIPT 03: TIDYVERSE DATA INGESTION, TEMPORAL NORMALIZATION, & NLP CLEANING (R)
# Task: Ingest 'apwa_parsed_archive.jsonl' natively in R with fault tolerance,
#       normalize 'clean_year', execute Tidyverse NLP cleaning (tokenization,
#       lemmatization, and the "Plural Bypass" expanded stop-word filter), and
#       export the synchronized STM mathematical matrix ('out').
# ==============================================================================

library(tidyverse)
library(jsonlite)
library(tidytext)
library(textstem)
library(stm)

# 1. Verify Workspace
safe_workspace <- "/content/drive/MyDrive/colab_apwa_folder"
if (getwd() != safe_workspace) setwd(safe_workspace)
cat("Operating inside R workspace:", getwd(), "\n")

# 2. Fault-Tolerant Line-by-Line JSONL Ingestion in R
# Rationale: Native stream_in() halts if a single line has a partial network write
# (such as line 2,650). Using read_lines() + purrr::safely() isolates and skips
# any corrupted JSON packet while keeping the pipeline 100% inside R/Tidyverse.
cat("Reading raw JSONL archive with fault tolerance...\n")
raw_lines <- read_lines("apwa_parsed_archive.jsonl")

safe_json_parse <- safely(function(x) {
  parsed <- fromJSON(x, simplifyVector = TRUE)
  tibble(
    url        = parsed$url %||% NA_character_,
    slug       = parsed$slug %||% NA_character_,
    transcript = parsed$transcript %||% NA_character_,
    date_raw   = parsed$metadata[["Date"]] %||% NA_character_
  )
})

parsed_results <- map(raw_lines, safe_json_parse)

# Log any skipped corrupted lines
failed_lines <- which(!map_lgl(parsed_results, ~ is.null(.x$error)))
if (length(failed_lines) > 0) {
  cat("Bypassed corrupted JSON records on line(s):", paste(failed_lines, collapse = ", "), "\n")
}

raw_df <- parsed_results %>%
  map("result") %>%
  compact() %>%
  bind_rows() %>%
  distinct(slug, .keep_all = TRUE)

cat("Successfully loaded", nrow(raw_df), "valid transcribed documents.\n")

# 3. Temporal Normalization (Extracting 4-Digit Authorship Year)
# Rationale: STM prevalence regression requires a clean numeric year covariate.
clean_data <- raw_df %>%
  mutate(
    clean_year = as.integer(str_extract(date_raw, "\\b(19|20)\\d{2}\\b"))
  ) %>%
  filter(!is.na(clean_year), !is.na(transcript), str_squish(transcript) != "") %>%
  select(slug, url, clean_year, date_raw, transcript)

cat("Dropped", nrow(raw_df) - nrow(clean_data), "document(s) missing a verifiable 4-digit year.\n")
cat("Temporally normalized corpus size:", nrow(clean_data), "documents.\n")

# 4. Expanded Domain-Specific Stop-Word Lexicon
# Rationale: High-entropy domain words ('prison', 'inmate') and conversational
# filler ('time', 'like', 'know') cause severe cross-topic contamination and
# degrade FREX exclusivity if not aggressively pruned.
custom_stops <- tibble(
  word = c(
    "prison", "inmate", "prisoner", "cell", "guard", "officer", "c.o.", "co",
    "apwa", "criminal", "crime", "life", "time", "like", "know", "get",
    "people", "day", "year", "way", "tell", "guy", "john", "hopkins", "university"
  )
)

# 5. Tidyverse NLP Pipeline & The "Plural Bypass" Fix
# Rationale: Evaluating tokens against the stop-word list both BEFORE and AFTER
# lemmatization ensures pluralized domain noise (e.g., "prisoners" -> "prisoner",
# "inmates" -> "inmate") cannot bypass the filter.
cat("Executing Tidytext tokenization, lemmatization, and Plural Bypass filtering...\n")

cleaned_corpus <- clean_data %>%
  # Step 5A: Tokenize into lowercase words, stripping punctuation
  unnest_tokens(output = word, input = transcript, token = "words", to_lower = TRUE) %>%
  # Step 5B: Keep strictly alphabetic tokens of at least 3 characters
  filter(str_detect(word, "^[a-z]{3,}$")) %>%
  # Step 5C: Pre-lemmatization stop-word removal (Standard + Custom)
  anti_join(stop_words, by = "word") %>%
  anti_join(custom_stops, by = "word") %>%
  # Step 5D: Lemmatize words to base morphological root via textstem
  mutate(lemma = tolower(lemmatize_words(word))) %>%
  # Step 5E: THE PLURAL BYPASS FIX (Post-lemmatization stop-word & length filter)
  filter(str_detect(lemma, "^[a-z]{3,}$")) %>%
  anti_join(stop_words, by = c("lemma" = "word")) %>%
  anti_join(custom_stops, by = c("lemma" = "word")) %>%
  # Step 5F: Re-aggregate cleaned lemmas per document while preserving metadata order
  group_by(slug) %>%
  summarize(cleaned_text = str_c(lemma, collapse = " "), .groups = "drop") %>%
  inner_join(clean_data, by = "slug") %>%
  select(slug, url, clean_year, date_raw, transcript, cleaned_text)

# 6. Build the Synchronized STM Mathematical Matrix ('out')
# Rationale: Passing our Tidy-cleaned text into textProcessor (with default R
# cleaning disabled) and prepDocuments() removes hapax legomena (single-use terms)
# and guarantees 1:1 row alignment between out$documents, out$meta, and raw transcripts.
cat("Constructing STM document-term matrix and pruning hapax legomena...\n")

processed <- textProcessor(
  documents         = cleaned_corpus$cleaned_text,
  metadata          = as.data.frame(cleaned_corpus),
  lowercase         = FALSE, # Handled in Tidyverse
  removestopwords   = FALSE, # Handled in Tidyverse
  removenumbers     = FALSE, # Handled in Tidyverse
  removepunctuation = FALSE, # Handled in Tidyverse
  stem              = FALSE, # Preserves textstem lemmatization
  wordLengths       = c(3, Inf)
)

# lower.thresh = 1 purges words appearing in only 1 document across the archive
out <- prepDocuments(
  documents    = processed$documents,
  vocab        = processed$vocab,
  meta         = processed$meta,
  lower.thresh = 1
)

# 7. Serialize Cleaned Data & STM Input Object to Disk
saveRDS(cleaned_corpus, file = "apwa_clean_corpus.rds")
saveRDS(out, file = "apwa_stm_out_object.rds")

cat("\n[TASK 03 COMPLETE] Saved 'apwa_clean_corpus.rds' and 'apwa_stm_out_object.rds'.\n")

## 04. STM Hyperparameter Tuning & `searchK` Diagnostics (R)
*Source: `scripts/04_stm_searchK_diagnostics.R`*  
Executes `searchK()` across $K \in \{10, 20, 30, 40, 50, 60\}$ to empirically justify selecting $K = 40$ based on Held-Out Likelihood, Residuals, Semantic Coherence, and Lower Bound.

In [ ]:
%%R
# ==============================================================================
# SCRIPT 04: STM HYPERPARAMETER TUNING & SEARCHK DIAGNOSTICS (R)
# Task: Execute searchK() across K = {10, 20, 30, 40, 50, 60} to empirically
#       justify selecting K = 40 based on Held-Out Likelihood, Residuals,
#       Semantic Coherence, and Lower Bound. Exports a Tidy ggplot2 figure.
# ==============================================================================

library(tidyverse)
library(stm)

# 1. Verify Workspace & Load Synchronized STM Corpus
safe_workspace <- "/content/drive/MyDrive/colab_apwa_folder"
if (getwd() != safe_workspace) setwd(safe_workspace)

out <- readRDS("apwa_stm_out_object.rds")
cat("Loaded STM corpus:", length(out$documents), "documents and", length(out$vocab), "vocab terms.\n")

# 2. Execute searchK Grid Search across Macroscopic Range
# Rationale: K < 30 collapses niche technology signals into general prison "soup",
# whereas K > 50 fragments coherent themes into statistical "crumbs".
k_grid <- c(10, 20, 30, 40, 50, 60)

cat("Beginning searchK evaluation across K =", paste(k_grid, collapse = ", "), "...\n")
set.seed(42)

search_results <- searchK(
  documents  = out$documents,
  vocab      = out$vocab,
  K          = k_grid,
  prevalence = ~ s(clean_year),
  data       = out$meta,
  init.type  = "Spectral",
  cores      = 2
)

# Immediately serialize searchK results to Google Drive
saveRDS(search_results, file = "apwa_searchK_results.rds")
cat("Saved 'apwa_searchK_results.rds' to Drive.\n")

# 3. Extract Metrics into a Tidy Tibble & Plot Publication-Ready Diagnostics
sk_df <- tibble(
  K                                = unlist(search_results$results$K),
  `Held-Out Likelihood (Maximize)` = unlist(search_results$results$heldout),
  `Residuals (Minimize)`           = unlist(search_results$results$residual),
  `Semantic Coherence (Maximize)`  = unlist(search_results$results$semcoh),
  `Lower Bound (Maximize)`         = unlist(search_results$results$lbound)
) %>%
  pivot_longer(cols = -K, names_to = "Metric", values_to = "Value")

p_searchk <- ggplot(sk_df, aes(x = K, y = Value)) +
  geom_line(linewidth = 1, color = "#2c3e50") +
  geom_point(size = 2.8, color = "#2c3e50") +
  geom_vline(xintercept = 40, linetype = "dashed", color = "#e74c3c", linewidth = 0.8) +
  facet_wrap(~ Metric, scales = "free_y", ncol = 2) +
  scale_x_continuous(breaks = k_grid) +
  labs(
    title    = "Structural Topic Model Diagnostic Metrics Across K = {10..60}",
    subtitle = "Red dashed line marks optimal dimensionality inflection point at K = 40",
    x        = "Number of Latent Topics (K)",
    y        = "Diagnostic Metric Value"
  ) +
  theme_bw(base_size = 12) +
  theme(
    strip.background = element_rect(fill = "#ecf0f1"),
    strip.text       = element_text(face = "bold"),
    plot.title       = element_text(face = "bold")
  )

ggsave("fig01_searchK_diagnostics.png", plot = p_searchk, width = 9, height = 6, dpi = 300)
print(p_searchk)

cat("\n[TASK 04 COMPLETE] Diagnostic plot saved to 'fig01_searchK_diagnostics.png'.\n")

## 05. Train Final $K = 40$ Structural Topic Model (R)
*Source: `scripts/05_stm_model_training_k40.R`*  
Trains the deterministic $K = 40$ STM using `Spectral` initialization with `~ s(clean_year)` prevalence prior and serializes the converged model to `apwa_stm_model_k40_final.rds`.

In [ ]:
%%R
# ==============================================================================
# SCRIPT 05: TRAIN FINAL K = 40 STRUCTURAL TOPIC MODEL (R)
# Task: Train the deterministic K = 40 STM using Spectral initialization and
#       inject 'clean_year' into the topic prevalence prior distribution.
#       Serialize the converged model to 'apwa_stm_model_k40_final.rds'.
# ==============================================================================

library(tidyverse)
library(stm)

# 1. Verify Workspace & Load STM Matrix Object
safe_workspace <- "/content/drive/MyDrive/colab_apwa_folder"
if (getwd() != safe_workspace) setwd(safe_workspace)

out <- readRDS("apwa_stm_out_object.rds")

# 2. Train the Final K = 40 Structural Topic Model
# Rationale: Spectral initialization is deterministic and guarantees global
# stability without sensitivity to random starting seeds.
cat("Training final K = 40 Structural Topic Model...\n")
set.seed(42)

final_model <- stm(
  documents  = out$documents,
  vocab      = out$vocab,
  K          = 40,
  prevalence = ~ s(clean_year),
  data       = out$meta,
  init.type  = "Spectral",
  verbose    = TRUE
)

# 3. Serialize Converged Model to Disk
saveRDS(final_model, file = "apwa_stm_model_k40_final.rds")

cat("\n[TASK 05 COMPLETE] Final K=40 model saved to 'apwa_stm_model_k40_final.rds'.\n")

## 06. Qualitative Topic Identification & Exemplar Validation (R)
*Source: `scripts/06_qualitative_validation.R`*  
Extracts top words (`Highest Prob`, `FREX`, `Lift`, `Score`) for all 40 topics into `top_topic_words_k40.csv` and runs `findThoughts()` on Topics 18, 32, and 37 to export un-lemmatized full-text exemplar essays into `qualitative_exemplar_essays.txt`.

In [ ]:
%%R
# ==============================================================================
# SCRIPT 06: QUALITATIVE TOPIC IDENTIFICATION & EXEMPLAR VALIDATION (R)
# Task: Extract top words (Highest Prob, FREX, Lift, Score) for all 40 topics
#       into a tidy CSV, and execute findThoughts() on Topics 18, 32, and 37
#       to pull un-lemmatized full-text essays for qualitative validation.
# ==============================================================================

library(tidyverse)
library(stm)

# 1. Verify Workspace & Load Model + Corpus
safe_workspace <- "/content/drive/MyDrive/colab_apwa_folder"
if (getwd() != safe_workspace) setwd(safe_workspace)

final_model <- readRDS("apwa_stm_model_k40_final.rds")
out         <- readRDS("apwa_stm_out_object.rds")

# 2. Extract Word Weighting Metrics (Highest Prob, FREX, Lift, Score)
# Rationale: FREX (Frequency and Exclusivity) balances how frequent a token is
# within a topic against how exclusive it is to that topic across the archive.
topic_labels <- labelTopics(final_model, n = 15)

tidy_topic_words <- tibble(
  topic_id     = 1:40,
  highest_prob = apply(topic_labels$prob, 1, paste, collapse = ", "),
  frex         = apply(topic_labels$frex, 1, paste, collapse = ", "),
  lift         = apply(topic_labels$lift, 1, paste, collapse = ", "),
  score        = apply(topic_labels$score, 1, paste, collapse = ", ")
)

write_csv(tidy_topic_words, "top_topic_words_k40.csv")
cat("Exported all 40 topic word-weighting profiles to 'top_topic_words_k40.csv'.\n\n")

# Print target topics to console for immediate inspection
target_topics <- c(18, 32, 37, 21, 26)
cat("=== TARGET TOPIC PROFILES (FREX & HIGHEST PROBABILITY) ===\n")
tidy_topic_words %>%
  filter(topic_id %in% target_topics) %>%
  select(topic_id, frex, highest_prob) %>%
  print(n = Inf)

# 3. Qualitative Validation via findThoughts()
# Rationale: Mitigates the "Sub-Topic Dilution Problem" by pulling the original,
# un-lemmatized transcripts of the top 5 highest-probability essays for each
# technology topic so the researcher can verify digital vs. analog context.
tech_topics <- c(
  "Topic 18: Interpersonal Connection (JPay/Kiosks/Tablets)" = 18,
  "Topic 32: Intellectual Utility (Tablets/Education)"       = 32,
  "Topic 37: State Surveillance (Digital Control/System)"    = 37
)

sink("qualitative_exemplar_essays.txt")
cat("======================================================================\n")
cat("APWA QUALITATIVE VALIDATION REPORT: TOP 5 EXEMPLAR ESSAYS PER TOPIC\n")
cat("======================================================================\n\n")

for (label in names(tech_topics)) {
  t_num <- tech_topics[[label]]
  cat("----------------------------------------------------------------------\n")
  cat(label, "\n")
  cat("FREX Words:", tidy_topic_words$frex[t_num], "\n")
  cat("----------------------------------------------------------------------\n\n")

  thoughts <- findThoughts(
    model  = final_model,
    texts  = out$meta$transcript,
    topics = t_num,
    n      = 5
  )

  doc_indices <- thoughts$index[[1]]
  for (rank_i in seq_along(doc_indices)) {
    idx <- doc_indices[rank_i]
    theta_val <- round(final_model$theta[idx, t_num], 4)
    cat(sprintf("[Rank %d | Slug: %s | Year: %d | Topic Proportion (Theta): %.4f]\n",
                rank_i, out$meta$slug[idx], out$meta$clean_year[idx], theta_val))
    cat("URL:", out$meta$url[idx], "\n\n")
    cat(str_sub(out$meta$transcript[idx], 1, 2500), "\n")
    cat("\n======================================================================\n\n")
  }
}
sink()

cat("\n[TASK 06 COMPLETE] Full-text exemplar essays written to 'qualitative_exemplar_essays.txt'.\n")

## 07. Temporal Covariate Estimation & Surge-Era Regressions (R)
*Source: `scripts/07_temporal_regressions.R`*  
Estimates B-spline temporal trajectories (`estimateEffect`) for Topics 18, 32, and 37, generates timeline plots (`1993–2024` and `2010–2024`), and runs OLS regressions over the broad decade (`2010–2021`) vs. the Prefigurative Surge Era (`2016–2021`).

In [ ]:
%%R
# ==============================================================================
# SCRIPT 07: TEMPORAL COVARIATE ESTIMATION & SURGE-ERA REGRESSIONS (R)
# Task: 1. Run estimateEffect() with B-splines s(clean_year) for Topics 18, 32, 37.
#       2. Generate publication plots for full timeline and active era (2010-2024).
#       3. Extract the Theta matrix to test linear regressions over the broad
#          decade (2010-2021) vs. the Prefigurative Surge Era (2016-2021).
#       4. Export publication-ready regression tables via modelsummary & gt.
# ==============================================================================

library(tidyverse)
library(stm)
library(broom)
library(modelsummary)
library(gt)

# 1. Verify Workspace & Load Model + Metadata
safe_workspace <- "/content/drive/MyDrive/colab_apwa_folder"
if (getwd() != safe_workspace) setwd(safe_workspace)

final_model <- readRDS("apwa_stm_model_k40_final.rds")
out         <- readRDS("apwa_stm_out_object.rds")

# Define target Tri-Partite topic indices
T_CONNECTION   <- 18
T_UTILITY      <- 32
T_SURVEILLANCE <- 37

# 2. Estimate Temporal Prevalence Effects via Smoothing Splines
# Rationale: Technology adoption in correctional facilities is non-linear;
# s(clean_year) allows trajectories to bend and capture sudden rollouts.
cat("Estimating B-spline temporal prevalence trajectories (Global uncertainty)...\n")

prep_tripartite <- estimateEffect(
  formula     = c(T_CONNECTION, T_UTILITY, T_SURVEILLANCE) ~ s(clean_year),
  stmobj      = final_model,
  metadata    = out$meta,
  uncertainty = "Global"
)

saveRDS(prep_tripartite, file = "apwa_prep_tripartite_effects.rds")

# 3. Plot A: Full Archive Timeline (1993-2024)
png("fig02_tripartite_trajectories_full.png", width = 2400, height = 2400, res = 300)
plot(
  prep_tripartite,
  covariate   = "clean_year",
  model       = final_model,
  topics      = c(T_CONNECTION, T_UTILITY, T_SURVEILLANCE),
  method      = "continuous",
  xlab        = "Year of Authorship",
  ylab        = "Expected Topic Proportion",
  main        = "Tri-Partite Trajectories of the Carceral-Digital Turn",
  ylim        = c(0.00, 0.15),
  linecol     = c("blue", "green3", "red"),
  printlegend = FALSE,
  ci.level    = 0.95
)
abline(v = 2021, col = "black", lty = 2, lwd = 1.8)
text(x = 2021, y = 0.14, labels = "2021 (Academic Turn)", pos = 2, cex = 0.85)
legend(
  "topleft",
  legend = c(
    "Interpersonal Connection (T18)",
    "Intellectual Utility (T32)",
    "State Surveillance (T37)"
  ),
  col = c("blue", "green3", "red"),
  lty = 1,
  lwd = 2,
  bty = "o",
  bg  = "white"
)
dev.off()

# 4. Plot B: Cropped Active Era (2010-2024) Eliminating 1990s Sparse-Data CIs
png("fig03_tripartite_trajectories_2010_2024.png", width = 2400, height = 2400, res = 300)
plot(
  prep_tripartite,
  covariate   = "clean_year",
  model       = final_model,
  topics      = c(T_CONNECTION, T_UTILITY, T_SURVEILLANCE),
  method      = "continuous",
  xlim        = c(2010, 2024),
  ylim        = c(0.00, 0.10),
  xlab        = "Year of Authorship",
  ylab        = "Expected Topic Proportion",
  main        = "Tri-Partite Trajectories (2010-2024)",
  linecol     = c("blue", "green3", "red"),
  printlegend = FALSE,
  ci.level    = 0.95
)
abline(v = 2021, col = "black", lty = 2, lwd = 1.8)
legend(
  "topleft",
  legend = c("Connection (T18)", "Utility (T32)", "Surveillance (T37)"),
  col    = c("blue", "green3", "red"),
  lty    = 1,
  lwd    = 2,
  bty    = "o",
  bg     = "white"
)
dev.off()

# 5. Era-Specific Linear Regressions on the Document-Topic Matrix (Theta)
# Rationale: Because T37 (Surveillance) exhibits a non-linear early hump in
# 2010-2015 before dipping and surging in 2016-2021, we compare the full decade
# window (2010-2021) against the isolated "Prefigurative Surge Era" (2016-2021).
theta_df <- tibble(
  slug             = out$meta$slug,
  Year             = out$meta$clean_year,
  Connection_T18   = final_model$theta[, T_CONNECTION],
  Utility_T32      = final_model$theta[, T_UTILITY],
  Surveillance_T37 = final_model$theta[, T_SURVEILLANCE]
)

decade_df <- theta_df %>% filter(Year >= 2010, Year <= 2021)
surge_df  <- theta_df %>% filter(Year >= 2016, Year <= 2021)

models_list <- list(
  "T18 Connection (2010-2021)"   = lm(Connection_T18 ~ Year, data = decade_df),
  "T32 Utility (2010-2021)"      = lm(Utility_T32 ~ Year, data = decade_df),
  "T37 Surveillance (2010-2021)" = lm(Surveillance_T37 ~ Year, data = decade_df),
  "T18 Connection (2016-2021)"   = lm(Connection_T18 ~ Year, data = surge_df),
  "T32 Utility (2016-2021)"      = lm(Utility_T32 ~ Year, data = surge_df),
  "T37 Surveillance (2016-2021)" = lm(Surveillance_T37 ~ Year, data = surge_df)
)

# Export tidy regression summary CSV
tidy_reg_summary <- map_dfr(models_list, ~ tidy(.x, conf.int = TRUE), .id = "Model") %>%
  left_join(
    map_dfr(models_list, ~ glance(.x) %>% select(r.squared, statistic, p.value, nobs), .id = "Model"),
    by = "Model",
    suffix = c("_coef", "_model_F")
  )

write_csv(tidy_reg_summary, "table01_prefigurative_regressions.csv")

# Export publication-ready HTML table via modelsummary + gt
ms_table <- modelsummary(
  models_list,
  stars   = c("*" = 0.05, "**" = 0.01, "***" = 0.001),
  gof_map = c("nobs", "r.squared", "F", "rmse"),
  title   = "Table 1: OLS Temporal Regressions of Topic Prevalence (theta) Prior to the 2021 Academic Turn",
  output  = "gt"
)

gtsave(ms_table, filename = "table01_prefigurative_regressions.html")

cat("\n[TASK 07 COMPLETE] Saved trajectory plots and publication regression tables.\n")

## 08. The Pandemic Catalyst ("Ratchet Effect") & $\theta$ Correlation (R)
*Source: `scripts/08_pandemic_catalyst.R`*  
Plots Topic 32 (Intellectual Utility) against Topic 21 (COVID-19 / Quarantine) from 2018 to 2024 to demonstrate the infrastructural "Ratchet Effect" following the 2020 lockdowns, and runs a document-level Pearson correlation test on the $\theta$ matrix.

In [ ]:
%%R
# ==============================================================================
# SCRIPT 08: THE PANDEMIC CATALYST ("RATCHET EFFECT") & THETA CORRELATION (R)
# Task: 1. Regress and plot Topic 32 (Intellectual Utility) alongside Topic 21
#          (COVID-19 / Quarantine) from 2018 to 2024 to demonstrate the
#          infrastructural "Ratchet Effect" following the 2020 lockdowns.
#       2. Execute a Pearson correlation test on the document-level Theta matrix
#          to test micro-document exclusivity between T21 and T32.
# ==============================================================================

library(tidyverse)
library(stm)
library(broom)

# 1. Verify Workspace & Load Model + Metadata
safe_workspace <- "/content/drive/MyDrive/colab_apwa_folder"
if (getwd() != safe_workspace) setwd(safe_workspace)

final_model <- readRDS("apwa_stm_model_k40_final.rds")
out         <- readRDS("apwa_stm_out_object.rds")

T_COVID   <- 21
T_UTILITY <- 32

# 2. Macro-Temporal Trajectory: T32 (Utility) vs. T21 (COVID-19)
cat("Estimating temporal trajectories for COVID-19 (T21) and Digital Utility (T32)...\n")

prep_covid <- estimateEffect(
  formula     = c(T_UTILITY, T_COVID) ~ s(clean_year),
  stmobj      = final_model,
  metadata    = out$meta,
  uncertainty = "Global"
)

png("fig04_pandemic_catalyst_ratchet_effect.png", width = 2400, height = 2400, res = 300)
plot(
  prep_covid,
  covariate   = "clean_year",
  model       = final_model,
  topics      = c(T_UTILITY, T_COVID),
  method      = "continuous",
  xlim        = c(2018, 2024),
  ylim        = c(0.00, 0.15),
  xlab        = "Year of Authorship",
  ylab        = "Expected Topic Proportion",
  main        = "The Pandemic Catalyst: Digital Utility vs. COVID-19",
  linecol     = c("green3", "darkorange"),
  printlegend = FALSE,
  ci.level    = 0.95
)
abline(v = 2020, col = "black", lty = 2, lwd = 1.8)
text(x = 2020, y = 0.14, labels = "2020 (Lockdowns)", pos = 4, cex = 0.85)
legend(
  "topleft",
  legend = c("Intellectual Utility / Tablets (T32)", "COVID-19 / Quarantine (T21)"),
  col    = c("green3", "darkorange"),
  lty    = 1,
  lwd    = 2,
  bty    = "o",
  bg     = "white"
)
dev.off()

# 3. Micro-Document Exclusivity: Pearson Correlation on the Theta Matrix
# Rationale: Tests whether writers blend COVID-19 and Digital Utility within the
# same essays (positive correlation) or whether the archive split concurrently
# into mutually exclusive response domains (negative correlation).
theta_matrix <- final_model$theta

cor_covid_utility <- cor.test(
  x      = theta_matrix[, T_COVID],
  y      = theta_matrix[, T_UTILITY],
  method = "pearson"
)

tidy_cor <- tidy(cor_covid_utility) %>%
  mutate(comparison = "Topic 21 (COVID-19) vs. Topic 32 (Intellectual Utility)") %>%
  select(comparison, estimate, statistic, p.value, parameter, conf.low, conf.high, method)

write_csv(tidy_cor, "table02_covid_utility_correlation.csv")

cat("\n=== MICRO-DOCUMENT EXCLUSIVITY TEST (T21 vs. T32) ===\n")
print(tidy_cor)

cat("\n[TASK 08 COMPLETE] Saved 'fig04_pandemic_catalyst_ratchet_effect.png' and correlation table.\n")

## 09. Topic Correlation Network & The Economy of Scarcity (R / `igraph`)
*Source: `scripts/09_network_analysis_igraph.R`*  
Computes the MAP topic correlation network across all 40 topics (`stm::topicCorr(cutoff = 0.05)`), renders the `igraph` network isolating the Topic 18 (Connection) $\leftrightarrow$ Topic 26 (Commissary/Food) tie, and verifies the Pearson correlation.

In [ ]:
%%R
# ==============================================================================
# SCRIPT 09: TOPIC CORRELATION NETWORK & THE ECONOMY OF SCARCITY (R / IGRAPH)
# Task: 1. Compute the MAP topic correlation network across all 40 topics using
#          stm::topicCorr(cutoff = 0.05).
#       2. Visualize the network via igraph, highlighting our target topics to
#          demonstrate both the compartmentalization of tech discourses and the
#          T18 (Connection) <-> T26 (Commissary/Food) "Economy of Scarcity" link.
# ==============================================================================

library(tidyverse)
library(stm)
library(igraph)
library(broom)

# 1. Verify Workspace & Load Final Model
safe_workspace <- "/content/drive/MyDrive/colab_apwa_folder"
if (getwd() != safe_workspace) setwd(safe_workspace)

final_model <- readRDS("apwa_stm_model_k40_final.rds")

# 2. Estimate Topic Correlation Matrix
# Rationale: Calculates positive covariance across document-topic proportions.
cormat <- topicCorr(final_model, method = "simple", cutoff = 0.05)
saveRDS(cormat, file = "apwa_topic_correlations.rds")

# 3. Export Standard STM Network Graph (Replication Benchmark)
png("fig05a_topic_correlation_network_stm.png", width = 2400, height = 2400, res = 300)
plot(
  cormat,
  vertex.color     = "lightblue",
  vertex.label.cex = 0.85,
  main             = "Topic Correlation Network (K=40)"
)
dev.off()

# 4. Build Enhanced Publication Network Graph via igraph
# Highlighting Target Nodes: T18 (Connection), T32 (Utility), T37 (Surveillance),
# T21 (COVID-19), and T26 (Commissary/Food).
g <- graph_from_adjacency_matrix(cormat$posadj, mode = "undirected", diag = FALSE)
V(g)$name <- paste0("Topic ", 1:40)

# Color-code target nodes vs. general archive topics
node_colors <- rep("#b2bec3", 40)
node_colors[18] <- "#3498db" # Interpersonal Connection (Blue)
node_colors[32] <- "#2ecc71" # Intellectual Utility (Green)
node_colors[37] <- "#e74c3c" # State Surveillance (Red)
node_colors[21] <- "#f39c12" # COVID-19 / Quarantine (Orange)
node_colors[26] <- "#9b59b6" # Commissary / Food (Purple)

V(g)$color <- node_colors

png("fig05b_topic_correlation_network_igraph.png", width = 2600, height = 2600, res = 300)
set.seed(42)
plot(
  g,
  layout             = layout_with_fr(g),
  vertex.size        = 13,
  vertex.color       = V(g)$color,
  vertex.frame.color = "#2d3436",
  vertex.label.cex   = 0.75,
  vertex.label.color = "black",
  vertex.label.font  = 2,
  edge.lty           = 2,
  edge.width         = 1.8,
  edge.color         = "#636e72",
  main               = "APWA Topic Correlation Network (K=40): Isolating the Economy of Scarcity"
)
legend(
  "bottomleft",
  legend = c(
    "T18: Interpersonal Connection",
    "T26: Commissary / Food (Linked to T18)",
    "T32: Intellectual Utility (Isolated)",
    "T37: State Surveillance (Isolated)",
    "T21: COVID-19 / Quarantine (Isolated)",
    "Other Archive Topics"
  ),
  pt.bg  = c("#3498db", "#9b59b6", "#2ecc71", "#e74c3c", "#f39c12", "#b2bec3"),
  pch    = 21,
  pt.cex = 1.8,
  cex    = 0.85,
  bty    = "n"
)
dev.off()

# 5. Direct Statistical Verification of the T18 (Connection) <-> T26 (Commissary) Tie
cor_scarcity <- cor.test(
  x      = final_model$theta[, 18],
  y      = final_model$theta[, 26],
  method = "pearson"
)

tidy_scarcity <- tidy(cor_scarcity) %>%
  mutate(comparison = "Topic 18 (Interpersonal Connection) vs. Topic 26 (Commissary/Food)") %>%
  select(comparison, estimate, statistic, p.value, parameter, conf.low, conf.high)

write_csv(tidy_scarcity, "table03_economy_of_scarcity_correlation.csv")

cat("\n=== ECONOMY OF SCARCITY LINK (T18 vs. T26) ===\n")
print(tidy_scarcity)

cat("\n[TASK 09 COMPLETE] All pipeline scripts, figures, and tables finished!\n")

## 10. Organize Output Artifacts & Export R `sessionInfo()`
*Source: `run_pipeline.py`*  
Routes generated data, tables, and figures into structured `data/`, `results/charts/`, and `results/figures/` directories and records the R environment specifications in `results/session_info.txt`.

In [ ]:
# ==============================================================================
# STEP 10A: ROUTE ARTIFACTS INTO STRUCTURED SUBFOLDERS (PYTHON)
# ==============================================================================

import os
import shutil

safe_workspace = '/content/drive/MyDrive/colab_apwa_folder'
os.chdir(safe_workspace)

for folder in ['data', 'results/charts', 'results/figures']:
    os.makedirs(os.path.join(safe_workspace, folder), exist_ok=True)

routing_map = {
    'data': [
        'apwa_master_urls.csv', 'apwa_parsed_archive.jsonl', 'apwa_excluded_urls.csv',
        'apwa_clean_corpus.rds', 'apwa_stm_out_object.rds', 'apwa_searchK_results.rds',
        'apwa_stm_model_k40_final.rds', 'apwa_prep_tripartite_effects.rds',
        'apwa_topic_correlations.rds'
    ],
    'results/charts': [
        'top_topic_words_k40.csv', 'qualitative_exemplar_essays.txt',
        'table01_prefigurative_regressions.csv', 'table01_prefigurative_regressions.html',
        'table02_covid_utility_correlation.csv', 'table03_economy_of_scarcity_correlation.csv'
    ],
    'results/figures': [
        'fig01_searchK_diagnostics.png', 'fig02_tripartite_trajectories_full.png',
        'fig03_tripartite_trajectories_2010_2024.png',
        'fig04_pandemic_catalyst_ratchet_effect.png',
        'fig05a_topic_correlation_network_stm.png',
        'fig05b_topic_correlation_network_igraph.png'
    ]
}

for target_dir, files in routing_map.items():
    for filename in files:
        if os.path.exists(filename):
            dest = os.path.join(target_dir, filename)
            shutil.copy2(filename, dest)
            print(f"Copied '{filename}' -> '{dest}'")

In [ ]:
%%R
# ==============================================================================
# STEP 10B: EXPORT R SESSION INFO FOR COMPUTATIONAL REPRODUCIBILITY (R)
# ==============================================================================

writeLines(capture.output(sessionInfo()), "results/session_info.txt")
cat("Exported R environment specifications to 'results/session_info.txt'.\n")